# Notebook 2 — GRPO Training

Trains the 170K MLP policy using GRPO rewards from frozen SmolVLM2.
Mirrors HORNet's two-stage training strategy:
- **Stage 1** — action_antonym (short SSv2 clips, simple MCQ)
- **Stage 2** — navigation/vlnqa (longer videos, temporal reasoning)

**Prerequisites:** Run Notebook 1 first (models and components must be loaded).

**Expected runtime:** ~1-3 hours per stage on a single A100 GPU.

## 1. Config & Paths

In [1]:
import os, json, gc
import numpy as np

# ── Dataset paths (same layout as your existing benchmark) ──────────────
DATA_ROOT     = '/DATA/teaching/DL_group45/data_set/MVBench'
ACTION_JSON   = f'{DATA_ROOT}/json/action_antonym.json'
NAV_JSON      = f'{DATA_ROOT}/json/egocentric_navigation.json'
ACTION_VIDEOS = f'{DATA_ROOT}/video/ssv2_video'
NAV_VIDEOS    = f'{DATA_ROOT}/video/vlnqa'

# ── Checkpoint save paths ────────────────────────────────────────────────
CKPT_STAGE1 = 'grpo_mlp_stage1.pt'
CKPT_FINAL  = 'grpo_mlp_final.pt'

# ── Hyperparameters ──────────────────────────────────────────────────────
TOP_K          = 8    # frames selected per video
POOL_SIZE      = 32   # uniformly sampled pool (reduced from 50)
NUM_CANDIDATES = 8    # GRPO candidate subsets per training step (K)
HASH_THRESHOLD = 8    # pHash Hamming distance for deduplication

print('Config loaded')
print(f'  Pool size      : {POOL_SIZE} frames (after dedup: typically 15-22)')
print(f'  Selected top-k : {TOP_K} frames → SmolVLM2')
print(f'  GRPO candidates: {NUM_CANDIDATES} per training step')

Config loaded
  Pool size      : 32 frames (after dedup: typically 15-22)
  Selected top-k : 8 frames → SmolVLM2
  GRPO candidates: 8 per training step


## 2. Dataset Loader

Converts MVBench JSON format (same keys your benchmark already uses) into a flat list for training.

In [2]:
def load_mvbench_json(json_path: str, video_folder: str) -> list:
    """
    Load MVBench JSON and return list of training dicts.
    Skips samples where the video file does not exist on disk.
    Output dict keys match your existing benchmark exactly:
        video_path, question, candidates, answer
    """
    with open(json_path) as f:
        data = json.load(f)
    samples = []
    for item in data:
        vp = os.path.join(video_folder, item['video'])
        if not os.path.exists(vp): continue
        samples.append({'video_path': vp, 'question': item['question'],
                         'candidates': item['candidates'], 'answer': item['answer']})
    print(f'  {len(samples)} valid samples from {os.path.basename(json_path)}')
    return samples

action_data = load_mvbench_json(ACTION_JSON, ACTION_VIDEOS)
nav_data    = load_mvbench_json(NAV_JSON,    NAV_VIDEOS)
print(f'Total: {len(action_data)} action + {len(nav_data)} navigation = '
      f'{len(action_data)+len(nav_data)} training samples')

  200 valid samples from action_antonym.json
  200 valid samples from egocentric_navigation.json
Total: 200 action + 200 navigation = 400 training samples


## 3. Feature Extractor

Extracts (T, 513) features per video: CLIP embeddings + motion scores.  
pHash deduplication is applied before CLIP to reduce encoding time.

In [3]:
def extract_features_for_training(video_path: str, num_frames: int = 32,
                                    hash_threshold: int = 8):
    """
    Full feature extraction pipeline for one video.

    Steps:
      1. Uniform sample `num_frames` frames
      2. pHash deduplication  (survey §3)
      3. CLIP ViT-B/32 encoding → (T', 512)
      4. Motion score appended  (survey §1) → (T', 513)

    Returns:
        frames_pil  : list of PIL Images (deduplicated, length T')
        features    : np.ndarray (T', 513)
    """
    result = FrameSampler.sample_frames_uniform(video_path, num_frames)
    if result is None: return None, None

    frames_all = result['frames']

    # pHash deduplication
    kept_idx   = phash_deduplicate(frames_all, hash_threshold)
    frames_pil = [frames_all[i] for i in kept_idx]
    if len(frames_pil) == 0: return None, None

    # CLIP + motion
    features = get_clip_embeddings_with_motion(frames_pil)  # (T', 513)
    return frames_pil, features

print('extract_features_for_training ready')

extract_features_for_training ready


## 4. Single Training Step

In [4]:
from components import (
        FrameSampler, GRPOFrameSelector, grpo_loss, get_clip_embeddings,FrameSelectorMLP,load_models,
        get_clip_embeddings_with_motion, compute_mcq_reward, smolvlm_answer,compute_mcq_reward,phash_deduplicate,compute_motion_scores
    )
load_models()  # <-- this MUST run before anything else

# verify models loaded correctly
from components import clip_processor, clip_model, smolvlm_model
print("clip_processor:", type(clip_processor))
print("clip_model:", type(clip_model))
print("smolvlm_model:", type(smolvlm_model))

/home/teaching/miniconda3/envs/dl45/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 8847.80it/s]


SmolVLM2-500M loaded and frozen
CLIP ViT-B/32  loaded and frozen
clip_processor: <class 'transformers.models.clip.processing_clip.CLIPProcessor'>
clip_model: <class 'transformers.models.clip.modeling_clip.CLIPModel'>
smolvlm_model: <class 'transformers.models.smolvlm.modeling_smolvlm.SmolVLMForConditionalGeneration'>


In [5]:
import torch
from typing import List

def training_step(selector, optimizer, sample, top_k=8, num_candidates=8):
    frames_pil, features = extract_features_for_training(sample['video_path'])
    if features is None:
        return None

    T = len(frames_pil)
    top_k = min(top_k, T)  # never request more frames than available
    
    feats = torch.tensor(features, dtype=torch.float32).to(selector.device)
    
    if not torch.isfinite(feats).all():
        return None

    probs = selector.mlp(feats)

    with torch.no_grad():
        cands = selector.generate_candidates(probs.detach(), k=top_k,
                                              num_candidates=num_candidates)

    rewards, best = [], 0.0
    for mask_idx in cands:
        selected = [frames_pil[i] for i in mask_idx]
        try:
            pred = smolvlm_answer(selected, sample['question'], sample['candidates'])
            r    = compute_mcq_reward(pred, sample['answer'])
        except Exception as e:
            print(f'  [skip candidate] smolvlm_answer failed: {e}')
            r = 0.0
        rewards.append(r)
        best = max(best, r)

    if all(r == 0.0 for r in rewards):
        return None  # skip GRPO update if all rewards are zero

    optimizer.zero_grad()
    loss = grpo_loss(probs, cands, rewards, T)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(selector.mlp.parameters(), max_norm=1.0)
    optimizer.step()

    return {'loss': loss.item(), 'mean_reward': float(np.mean(rewards)),
            'best_reward': best}

print('training_step ready')

training_step ready


## 5. Full Training Loop

In [6]:
def train_grpo(selector: GRPOFrameSelector, dataset: list,
                epochs: int = 2, lr: float = 1e-4,
                top_k: int = 8, num_candidates: int = 8,
                save_path: str = 'grpo_mlp.pt', log_every: int = 20) -> GRPOFrameSelector:
    """
    Train the MLP policy with GRPO.
    SmolVLM2 and CLIP remain completely frozen.
    Only selector.mlp weights are updated.

    Args:
        selector       : GRPOFrameSelector (MLP to train)
        dataset        : list of dicts {video_path, question, candidates, answer}
        epochs         : training epochs
        lr             : Adam learning rate
        top_k          : frames to select per video
        num_candidates : GRPO candidate subsets per step (K=8 in HORNet)
        save_path      : checkpoint path
        log_every      : print progress every N steps
    """
    import sys
    selector.mlp.train()
    optimizer = torch.optim.Adam(selector.mlp.parameters(), lr=lr)
    N = len(dataset)

    for epoch in range(1, epochs + 1):
        epoch_loss = epoch_reward = epoch_acc = 0.0; skipped = 0

        for step, sample in enumerate(dataset, 1):
            result = training_step(selector, optimizer, sample, top_k, num_candidates)
            if result is None: skipped += 1; continue

            epoch_loss   += result['loss']
            epoch_reward += result['mean_reward']
            epoch_acc    += result['best_reward']

            if step % log_every == 0:
                n = step - skipped
                print(f'[Epoch {epoch}/{epochs}] Step {step}/{N} | '
                      f'Loss: {epoch_loss/n:.4f} | '
                      f'Avg Reward: {epoch_reward/n:.3f} | '
                      f'Best-of-K Acc: {epoch_acc/n:.3f} | '
                      f'Skipped: {skipped}')
                sys.stdout.flush()

        n = N - skipped
        print(f'\n[Epoch {epoch} done] '
              f'Loss: {epoch_loss/max(n,1):.4f} | '
              f'Reward: {epoch_reward/max(n,1):.3f}\n')
        ep_path = save_path.replace('.pt', f'_ep{epoch}.pt')
        torch.save(selector.mlp.state_dict(), ep_path)
        print(f'Checkpoint saved: {ep_path}')

    torch.save(selector.mlp.state_dict(), save_path)
    print(f'Final checkpoint: {save_path}')
    selector.mlp.eval()
    return selector

print('train_grpo ready')

train_grpo ready


## 6. Stage 1 — Train on action_antonym

Short SSv2 clips (2-10s). Trains MLP to identify action-relevant frames.  
Mirrors HORNet Stage 1 (MSVD/MSRVTT short-form QA).

In [7]:
# Step 1 - check CUDA before loading anything
import torch
print(torch.randn(3,3).cuda())  # if this fails, GPU hardware issue

tensor([[ 1.8764,  1.2997, -0.7450],
        [ 0.4305, -0.5860, -1.1594],
        [ 0.1880,  0.6008, -0.3351]], device='cuda:0')


In [8]:
print('=== STAGE 1: action_antonym ===')
selector = GRPOFrameSelector(input_dim=513)

selector = train_grpo(
    selector       = selector,
    dataset        = action_data,
    epochs         = 2,
    lr             = 1e-4,
    top_k          = TOP_K,
    num_candidates = NUM_CANDIDATES,
    save_path      = CKPT_STAGE1,
    log_every      = 20,
)

=== STAGE 1: action_antonym ===
[GRPOFrameSelector] MLP params: 148,097


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

[Epoch 1/2] Step 20/200 | Loss: 0.0000 | Avg Reward: 1.000 | Best-of-K Acc: 1.000 | Skipped: 8


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

[Epoch 1/2] Step 80/200 | Loss: -0.0000 | Avg Reward: 0.990 | Best-of-K Acc: 1.000 | Skipped: 28


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

... (19 outputs left out)

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

[Epoch 2/2] Step 180/200 | Loss: -0.0001 | Avg Reward: 0.982 | Best-of-K Acc: 1.000 | Skipped: 69


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

[Epoch 2/2] Step 200/200 | Loss: -0.0001 | Avg Reward: 0.984 | Best-of-K Acc: 1.000 | Skipped: 77

[Epoch 2 done] Loss: -0.0001 | Reward: 0.984

Checkpoint saved: grpo_mlp_stage1_ep2.pt
Final checkpoint: grpo_mlp_stage1.pt


## 7. Stage 2 — Fine-tune on navigation

Longer vlnqa videos, temporal reasoning required.  
This is where the -2% accuracy gap in your existing SMI pipeline is expected to recover.  
Mirrors HORNet Stage 2 (NExT-QA MCQ, +7.3 points over uniform).

In [9]:
print('=== STAGE 2: navigation (fine-tune) ===')

selector = train_grpo(
    selector       = selector,        # continue from stage 1
    dataset        = nav_data,
    epochs         = 3,
    lr             = 5e-5,            # lower LR for fine-tuning
    top_k          = TOP_K,
    num_candidates = NUM_CANDIDATES,
    save_path      = CKPT_FINAL,
    log_every      = 20,
)

print(f'Training complete. Final checkpoint: {CKPT_FINAL}')

=== STAGE 2: navigation (fine-tune) ===


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

[Epoch 1/3] Step 40/200 | Loss: 0.0002 | Avg Reward: 0.921 | Best-of-K Acc: 1.000 | Skipped: 21


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

[Epoch 1/3] Step 60/200 | Loss: 0.0002 | Avg Reward: 0.948 | Best-of-K Acc: 1.000 | Skipped: 31


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

... (31 outputs left out)

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

[Epoch 3/3] Step 140/200 | Loss: 0.0002 | Avg Reward: 0.973 | Best-of-K Acc: 1.000 | Skipped: 48


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i


[Epoch 3 done] Loss: 0.0002 | Reward: 0.975

Checkpoint saved: grpo_mlp_final_ep3.pt
Final checkpoint: grpo_mlp_final.pt
Training complete. Final checkpoint: grpo_mlp_final.pt
